# 00 · Explorar los datos

Cuenta estudios y series, revisa las 58 etiquetas gold y abre ejemplos DICOM por plano. Tarda unos 2–5 minutos, no necesita GPU y solo lee unas pocas series.

In [ ]:
# CONFIG
RUN_NAME = "exploracion"
RAW_ROOT = "/workspace/data/raw"
N_REPORTS = 3


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(repo))
from src.knee.config import LABELS
from src.knee.dicom import read_series

raw = Path(RAW_ROOT)
def read_train_csv(path):
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"No se pudo leer {path}")

train = read_train_csv(raw / "train.csv")
series = pd.read_csv(raw / "train_series.csv")


In [ ]:
gold = train.dropna(subset=LABELS, how="all")
print(f"Estudios: {len(train):,}")
print(f"Con etiquetas gold: {len(gold):,}")
print(f"Series: {len(series):,}")
display(series.groupby("StudyInstanceUID").size().describe())
display(series.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).size().rename("series"))
display(pd.DataFrame({"positivos": gold[LABELS].sum(), "negativos": len(gold) - gold[LABELS].sum()}))


In [ ]:
for _, row in train.sample(min(N_REPORTS, len(train)), random_state=42).iterrows():
    print("\nESTUDIO", row.StudyInstanceUID)
    print(str(row.Report)[:1200])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, plane in zip(axes, ["Sagittal", "Coronal", "Axial"]):
    row = series[series.Anatomical_Plane == plane].iloc[0]
    folder = raw / "train_series" / row.StudyInstanceUID / row.SeriesInstanceUID
    volume = read_series(folder)
    axis.imshow(volume[len(volume) // 2], cmap="gray")
    axis.set_title(f"{plane} · {len(volume)} cortes")
    axis.axis("off")
plt.tight_layout()
